In [2]:
# 현재 연결된 커널에 강제로 라이브러리 설치
!pip install langchain langchain-community langchain-upstage langchain-chroma chromadb python-dotenv python-docx requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 6.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 6.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 6.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 609.9/609.9 kB 6.4 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0.1 -> 26.0
[notice] To update, run: pip install --upgrade pip


In [1]:
import langchain
print("성공!")

성공!


In [2]:
import os
from dotenv import load_dotenv
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from langchain_community.vectorstores import Chroma
import chromadb

# .env 파일에 저장된 UPSTAGE_API_KEY 로드
load_dotenv()
UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")

# 모델 및 임베딩 초기화
llm = ChatUpstage(model="solar-pro", api_key=UPSTAGE_API_KEY)
embeddings = UpstageEmbeddings(model="solar-embedding-1-large", api_key=UPSTAGE_API_KEY)

print("✅ 환경 설정 완료!")

✅ 환경 설정 완료!


In [ ]:
# 도커 외부에서 접속하므로 4008 포트 사용
# 만약 WSL 내부 주피터라면 'localhost' 혹은 '127.0.0.1'
client = chromadb.HttpClient(host="127.0.0.1", port=4008)
# 컬렉션 생성 또는 로드
vectorstore = Chroma(
    client=client,
    collection_name="law_docs",
    embedding_function=embeddings
)

print("✅ ChromaDB 서버 연결 성공!")

ValueError: {"detail":"Not Found"}

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 문서 로드
loader = DirectoryLoader("./docs", glob="**/*.docx", loader_cls=Docx2txtLoader)
docs = loader.load()

# 문서 분할
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
splits = splitter.split_documents(docs)

# DB에 추가
vectorstore.add_documents(splits)

print(f"✅ {len(docs)}개 문서, {len(splits)}개 조각 학습 완료!")

In [ ]:
import requests

# 외부 포트 4007번 사용!
url = "http://localhost:4007/ask/law"
data = {"question": "전자상거래법상 청약철회 기간은 어떻게 돼?"}

response = requests.post(url, json=data)
print(response.json())